# MBG 5-Page Extraction Test (Qwen)
This notebook runs MBG extraction on the first 5 pages using precomputed OCR outputs.

In [ ]:
from pathlib import Path
import json
import subprocess

ROOT = Path.cwd()
NAMESPACE = "exp_mbg_v1"
OCR_DIR = ROOT / "artifacts" / "experiments" / NAMESPACE / "ocr" / "deepseek_fullbook"
OCR_PAGES_JSONL = OCR_DIR / "pages_ocr.jsonl"
TEST_DIR = ROOT / "artifacts" / "experiments" / NAMESPACE / "extraction" / "mbg_5page_qwen_test"
TEST_DIR.mkdir(parents=True, exist_ok=True)

LLM_ENDPOINT = "http://10.6.32.16:8000/v1"
LLM_MODEL = "nvidia/Qwen3.6-35B-A3B-NVFP4"

# Auto-pick 5 contiguous pages around the middle using existing OCR results.
page_numbers = []
with OCR_PAGES_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if row.get("error"):
            continue
        page = row.get("page_number")
        if isinstance(page, int):
            page_numbers.append(page)

page_numbers = sorted(set(page_numbers))
if len(page_numbers) < 5:
    raise RuntimeError("Not enough OCR pages to select 5 pages")

mid_start_idx = max(0, len(page_numbers) // 2 - 2)
selected_pages = page_numbers[mid_start_idx:mid_start_idx + 5]
if len(selected_pages) < 5:
    selected_pages = page_numbers[-5:]

PAGE_START = selected_pages[0]
PAGE_END = selected_pages[-1]
MBG_OUT = TEST_DIR / f"lb_mbg_cards_p{PAGE_START}_p{PAGE_END}.jsonl"

print("Namespace:", NAMESPACE)
print("OCR dir:", OCR_DIR)
print("OCR pages JSONL:", OCR_PAGES_JSONL)
print("Selected pages:", selected_pages)
print("Page range:", PAGE_START, "-", PAGE_END)
print("Output JSONL:", MBG_OUT)

OCR dir: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook
OCR pages JSONL: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook/pages_ocr.jsonl
Selected pages: [36, 37, 38, 39, 40]
Page range: 36 - 40
Output JSONL: /home/snt/projects/GramMetaRL/artifacts/mbg_5page_qwen_test/lb_mbg_cards_p36_p40.jsonl


In [21]:
# Confirm middle-page selection details
print('Middle 5-page extraction plan:')
print('  Start page:', PAGE_START)
print('  End page:', PAGE_END)
print('  Count:', PAGE_END - PAGE_START + 1)
print('  Source OCR path:', OCR_PAGES_JSONL)
print('  No OCR re-run. Using existing artifacts directly.')

Middle 5-page extraction plan:
  Start page: 36
  End page: 40
  Count: 5
  Source OCR path: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook/pages_ocr.jsonl
  No OCR re-run. Using existing artifacts directly.


In [ ]:
# Run MBG extraction with Qwen using existing OCR outputs only (no PDF input)
cmd = [
    "python",
    "main.py",
    "--namespace", NAMESPACE,
    "build-mbg",
    "--output", str(MBG_OUT),
    "--language", "lb",
    "--source-id", f"luxembourgish_grammar_book_p{PAGE_START}_p{PAGE_END}",
    "--ocr-pages-dir", str(OCR_DIR),
    "--page-start", str(PAGE_START),
    "--page-end", str(PAGE_END),
    "--llm-endpoint", LLM_ENDPOINT,
    "--llm-model", LLM_MODEL,
    "--llm-timeout", "600",
    "--max-chunk-chars", "2000",
]

print("Running command:")
print(" ".join(cmd))

proc = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

captured_lines = []
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
    captured_lines.append(line)

ret = proc.wait()
print("\nReturn code:", ret)
if ret != 0:
    tail = "".join(captured_lines[-120:])
    print("\n--- LAST LOG LINES ---")
    print(tail)
    raise RuntimeError("build-mbg failed")

Running command:
/home/snt/projects/GramMetaRL/.venv/bin/python /home/snt/projects/GramMetaRL/main.py build-mbg --output /home/snt/projects/GramMetaRL/artifacts/mbg_5page_qwen_test/lb_mbg_cards_p36_p40.jsonl --language lb --source-id luxembourgish_grammar_book_p36_p40 --ocr-pages-dir /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook --page-start 36 --page-end 40 --llm-endpoint http://10.6.32.16:8000/v1 --llm-model nvidia/Qwen3.6-35B-A3B-NVFP4 --llm-timeout 600 --max-chunk-chars 2000
[MBG] section 1/2 title=## 4.6 Prepositions pages=36-37
[MBG] section 1/2 extracted=6 added=6
[MBG] section 2/2 title=## 4.7 Verbs pages=37-40
[MBG] section 2/2 extracted=12 added=12
[MBG] completed sections=2 cards=18 output=/home/snt/projects/GramMetaRL/artifacts/mbg_5page_qwen_test/lb_mbg_cards_p36_p40.jsonl
Built 18 MBG cards -> /home/snt/projects/GramMetaRL/artifacts/mbg_5page_qwen_test/lb_mbg_cards_p36_p40.jsonl

Return code: 0


In [23]:
# Inspect extraction outputs
cards = []
with MBG_OUT.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            cards.append(json.loads(line))

print('Output JSONL:', MBG_OUT)
print('Total cards:', len(cards))

for i, c in enumerate(cards[:8], start=1):
    print(f'\n[{i}] id={c.get("id")}, scope={c.get("scope")}, op={c.get("operation_type")}, priority={c.get("priority")}')
    print('tags:', c.get('phenomenon_tags', []))
    print('triggers:', c.get('trigger_conditions', [])[:3])
    src = c.get('source', {})
    print('source pages:', src.get('page_start'), '-', src.get('page_end'), '| section:', src.get('section_title'))

Output JSONL: /home/snt/projects/GramMetaRL/artifacts/mbg_5page_qwen_test/lb_mbg_cards_p36_p40.jsonl
Total cards: 18

[1] id=LB_0001, scope=prepositional phrase | noun phrase complement | None, op=inflect, priority=60
tags: ['case_marking', 'preposition_governance']
triggers: ["{'condition': 'Preposition governs accusative and context expresses direction toward a location', 'required': True, 'feature_bundle': {'clause_type': None, 'constituent_type': 'prepositional phrase', 'tense_aspect_mood': None, 'polarity': None, 'person_number_gender': None, 'definiteness': None, 'register': None, 'dialect': None}}"]
source pages: 36 - 37 | section: ## 4.6 Prepositions

[2] id=LB_0002, scope=prepositional phrase | noun phrase complement | None, op=inflect, priority=60
tags: ['case_marking', 'preposition_governance']
triggers: ["{'condition': 'Preposition governs dative and context expresses static position', 'required': True, 'feature_bundle': {'clause_type': None, 'constituent_type': 'prepositio